In [3]:
models = [
    "qwen2.5vl:7b",
     'gemma3:12b',
    "llava:7b"]
    

In [4]:
from utils import *

In [5]:
#pip install ollama
import ollama

In [6]:
from pydantic import BaseModel, Field

class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str

In [7]:
class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str = Field(
        min_length=20,
        max_length=300,
        description="Brief coaching feedback in 2-4 sentences."
    )

In [8]:
throwprompt2 = """
You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = strong collegiate athlete
10 = world class technique

Give 2-4 sentences of specific, actionable feedback.
"""

throwprompt3 = """
You are a discus coach whose job is to identify the single biggest flaw in a throw.

Look across the full sequence of images and determine:
1. The overall score from 1-10
2. The biggest mistake hurting the throw
3. The one change that would improve the throw the most

Keep the feedback concise and concrete. Do not mention uncertainty unless absolutely necessary.
"""

throwprompt4 = """
You are an AI sports judge trained to score discus technique from still frames.

You are seeing multiple frames sampled throughout one throw. Infer the movement between frames.

Score the throw from 1-10 using this rubric:
- 1-3: major balance, timing, or release problems
- 4-6: decent throw with noticeable flaws
- 7-8: technically good throw with minor issues
- 9-10: highly polished, competitive form

In your feedback, mention at least:
- one thing the athlete did well
- one thing to improve
- what phase of the throw looked weakest
"""

throwprompt5 = """
You are a brutally honest but helpful throwing coach.

Analyze the series of images as if you are watching a video of the throw. Pay close attention to:
- whether the athlete stays balanced
- whether the hips lead the shoulders
- whether the throwing arm lags correctly
- whether the release appears high and powerful

Give a score from 1-10. Then provide feedback in the style of a coach talking directly to the athlete: short, blunt, and practical.
"""

In [9]:

throwprompt="""
You are an expert AI discus throw coach.

You have been provided a series images of a pupil throwing a discus start to finish. 

Rate the discus thrower's form 1-10 then give brief feeback.

"""

def critique_throw(image_paths: list,
                   mod: str = "llava:7b",
                   prompt: str = throwprompt,  
                   temp: float = 0.0):
    
    resp = ollama.chat(
        model=mod,
        messages=[
            {
                "role": "user",
                "content": prompt,
                "images": image_paths
            }
        ],
        format=ThrowCritique.model_json_schema(),
        options={"temperature": temp}
    )

    raw = resp["message"]["content"]
    return ThrowCritique.model_validate_json(raw)

In [10]:
prompts=[throwprompt,throwprompt2, throwprompt3,throwprompt4, throwprompt5]

In [11]:
import time
import pandas as pd

In [12]:
def doscore(count,instructions,mod):
    scores=[]
    durations=[]
    feedbacks=[]
    for i in range(1,8):
        if i==1:
            idstr=""
        else:
            idstr="-"+str(i)
    
        file=f"videoplayback{idstr}.mp4"
        start=time.time()
        frames = sample_frames_from_mp4(file,count)
        img_paths=write_temp_images_for_llm(frames)
        crit=critique_throw(image_paths=img_paths,mod=mod,prompt=instructions)
        stop=time.time()
    
        duration=stop-start
    
        durations.append(duration)
        scores.append(crit.score)
        
    return pd.DataFrame({'scores':scores,
                         "duration":durations}).assign(Frames=count).\
        assign(Prompt=instructions[:10]).assign(model=mod)

In [13]:
def doscore(count, instructions, mod):
    scores = []
    durations = []

    for i in range(1, 8):
        idstr = "" if i == 1 else f"-{i}"
        file = f"videoplayback{idstr}.mp4"
        start = time.time()

        try:
            frames = sample_frames_from_mp4(file, count)
            img_paths = write_temp_images_for_llm(frames)
            crit = critique_throw(image_paths=img_paths, mod=mod, prompt=instructions)
            scores.append(crit.score)
        except Exception as e:
            print(f"Failed on {file}: {e}")
            scores.append(pd.NA)

        durations.append(time.time() - start)

    return (
        pd.DataFrame({"scores": scores, "duration": durations})
        .assign(Frames=count, Prompt=instructions[:20], model=mod)
    )

In [ ]:
report=pd.DataFrame()
for i in range(3,16,3):
    for j in prompts:
        for k in models:
            df=doscore(i,j,k)
            report=pd.concat([report,df])
            print(i,k)

3 qwen2.5vl:7b


In [ ]:
report

In [ ]:
report.to_csv("zeroreport.csv")

In [ ]:
report.groupby("Frames").agg(AVGtime=('duration','mean'),
                             ScoreSTD=('scores','std'),
                            ScoreMean=('scores','mean'),
                            ScoreUnique=('scores','nunique')).\
    plot(marker="o")

In [ ]:
report.groupby("Frames").agg(AVGtime=('duration','mean'),
                             ScoreSTD=('scores','std'),
                            ScoreMean=('scores','mean'))